# GPU Accelerated Image Processing: Parallel Filtering & Transformation Pipelines (CUDA)
Runs on Google Colab (Runtime > Change runtime type > GPU). Uses Numba CUDA, so every
kernel is real CUDA (shared-memory tiling, coalesced access, atomics) but written in Python.
Pipeline: Grayscale -> Gaussian -> Sobel, plus Median 3x3 and Histogram Equalization.

In [ ]:
import os, time, math, urllib.request, subprocess, sys
import numpy as np
import numba as nb
from numba import cuda, float32, int32, prange
import matplotlib.pyplot as plt
from PIL import Image

if not cuda.is_available():
  subprocess.run([sys.executable, "-m", "pip", "-q", "install", "numba-cuda"])
  raise SystemExit("Installed numba-cuda. Go to Runtime > Restart Session, then re-run from Cell 1."
                    "If still fails, check Runtime > Change runtime types > T4 GPU.")
print("GPU: ", cuda.get_current_device().name)
print("CPU threads (OpenMP-style): ", nb.config.NUMBA_NUM_THREADS)

In [ ]:
KODAK_URLS = [
    "http://r0k.us/graphics/kodak/kodak/kodim{i:02d}.png",
    "https://r0k.us/graphics/kodak/kodak/kodim{i:02d}.png",
]
DATA_DIR = "kodak"
os.makedirs(DATA_DIR, exist_ok=True)
 
def download_kodak(n=24):
    paths = []
    for i in range(1, n + 1):
        p = f"{DATA_DIR}/kodim{i:02d}.png"
        if not os.path.exists(p):
            for tpl in KODAK_URLS:
                try:
                    req = urllib.request.Request(tpl.format(i=i), headers={"User-Agent": "Mozilla/5.0"})
                    with urllib.request.urlopen(req, timeout=20) as r, open(p, "wb") as f:
                        f.write(r.read())
                    break
                except Exception as e:
                    err = e
            else:
                print(f"kodim{i:02d} failed: {err}")
                continue
        paths.append(p)
    return paths
 
paths = download_kodak()
if not paths:  # offline fallback so the notebook still runs
    from skimage import data
    Image.fromarray(data.astronaut()).save(f"{DATA_DIR}/astronaut.png")
    paths = [f"{DATA_DIR}/astronaut.png"]
print(len(paths), "images ready")
 
def load_rgb(path, size=None):
    im = Image.open(path).convert("RGB")
    if size:  # size = (H, W)
        im = im.resize((size[1], size[0]), Image.BICUBIC)
    return np.ascontiguousarray(np.asarray(im, dtype=np.uint8))

In [ ]:
def make_cpu_ops(parallel):
    @nb.njit(parallel=parallel, fastmath=True)
    def gray(rgb):
        H, W, _ = rgb.shape
        out = np.empty((H, W), np.float32)
        for y in prange(H):
            for x in range(W):
                out[y, x] = 0.299 * rgb[y, x, 0] + 0.587 * rgb[y, x, 1] + 0.114 * rgb[y, x, 2]
        return out
 
    @nb.njit(parallel=parallel, fastmath=True)
    def conv(src, w, R):
        H, W = src.shape
        out = np.empty((H, W), np.float32)
        K = 2 * R + 1
        for y in prange(H):
            for x in range(W):
                acc = 0.0
                for dy in range(-R, R + 1):
                    yy = min(max(y + dy, 0), H - 1)
                    for dx in range(-R, R + 1):
                        xx = min(max(x + dx, 0), W - 1)
                        acc += src[yy, xx] * w[(dy + R) * K + dx + R]
                out[y, x] = acc
        return out
 
    @nb.njit(parallel=parallel, fastmath=True)
    def sobel(src):
        H, W = src.shape
        out = np.empty((H, W), np.float32)
        for y in prange(H):
            for x in range(W):
                gx = 0.0
                gy = 0.0
                for dy in range(-1, 2):
                    yy = min(max(y + dy, 0), H - 1)
                    for dx in range(-1, 2):
                        xx = min(max(x + dx, 0), W - 1)
                        v = src[yy, xx]
                        gx += v * dx * (2 if dy == 0 else 1)
                        gy += v * dy * (2 if dx == 0 else 1)
                out[y, x] = min(math.sqrt(gx * gx + gy * gy), 255.0)
        return out
 
    @nb.njit(parallel=parallel)
    def median3(src):
        H, W = src.shape
        out = np.empty((H, W), np.float32)
        for y in prange(H):
            win = np.empty(9, np.float32)
            for x in range(W):
                k = 0
                for dy in range(-1, 2):
                    yy = min(max(y + dy, 0), H - 1)
                    for dx in range(-1, 2):
                        xx = min(max(x + dx, 0), W - 1)
                        win[k] = src[yy, xx]
                        k += 1
                for i in range(1, 9):
                    v = win[i]
                    j = i - 1
                    while j >= 0 and win[j] > v:
                        win[j + 1] = win[j]
                        j -= 1
                    win[j + 1] = v
                out[y, x] = win[4]
        return out
 
    @nb.njit
    def hist_eq(src):  # histogram is a serial reduction on CPU
        H, W = src.shape
        hist = np.zeros(256, np.int64)
        for y in range(H):
            for x in range(W):
                hist[min(max(int(src[y, x]), 0), 255)] += 1
        cdf = np.cumsum(hist)
        cmin = 0
        for i in range(256):
            if cdf[i] > 0:
                cmin = cdf[i]
                break
        lut = np.empty(256, np.float32)
        for i in range(256):
            lut[i] = max(0.0, (cdf[i] - cmin)) / max(1, H * W - cmin) * 255.0
        out = np.empty((H, W), np.float32)
        for y in range(H):
            for x in range(W):
                out[y, x] = lut[min(max(int(src[y, x]), 0), 255)]
        return out
 
    return gray, conv, sobel, median3, hist_eq
 
CPU_SEQ = make_cpu_ops(False)
CPU_OMP = make_cpu_ops(True)
 
def gaussian_weights(R, sigma=None):
    sigma = sigma or max(R / 2.0, 0.8)
    ax = np.arange(-R, R + 1, dtype=np.float32)
    g = np.exp(-(ax[:, None] ** 2 + ax[None, :] ** 2) / (2 * sigma ** 2))
    return (g / g.sum()).astype(np.float32).ravel()
 
def run_cpu(ops, rgb, R=2):
    gray, conv, sobel, median3, hist_eq = ops
    w = gaussian_weights(R)
    g = gray(rgb)
    blur = conv(g, w, R)
    edges = sobel(blur)
    med = median3(g)
    heq = hist_eq(g)
    return dict(gray=g, blur=blur, edges=edges, median=med, heq=heq)

In [ ]:
TILE = 16
TPB = (TILE, TILE)
SH = TILE + 2          # shared-memory tile edge for 3x3 kernels (must be a plain constant)
 
@cuda.jit
def k_gray(rgb, out, H, W):
    x, y = cuda.grid(2)  # x is the fastest axis so warps read consecutive pixels (coalesced)
    if x < W and y < H:
        out[y, x] = 0.299 * rgb[y, x, 0] + 0.587 * rgb[y, x, 1] + 0.114 * rgb[y, x, 2]
 
def make_conv_kernel(R):
    """Shared-memory tiled KxK convolution. R is a compile-time constant."""
    S = TILE + 2 * R
    K = 2 * R + 1
 
    @cuda.jit
    def k(src, dst, w, H, W):
        tile = cuda.shared.array((S, S), float32)
        tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
        x0 = cuda.blockIdx.x * TILE - R
        y0 = cuda.blockIdx.y * TILE - R
        for j in range(ty, S, TILE):  # cooperative load incl. halo, clamped at borders
            yy = min(max(y0 + j, 0), H - 1)
            for i in range(tx, S, TILE):
                xx = min(max(x0 + i, 0), W - 1)
                tile[j, i] = src[yy, xx]
        cuda.syncthreads()
        x = cuda.blockIdx.x * TILE + tx
        y = cuda.blockIdx.y * TILE + ty
        if x < W and y < H:
            acc = 0.0
            for dy in range(K):
                for dx in range(K):
                    acc += tile[ty + dy, tx + dx] * w[dy * K + dx]
            dst[y, x] = acc
    return k
 
@cuda.jit
def k_sobel(src, dst, H, W):
    tile = cuda.shared.array((SH, SH), float32)
    tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
    x0 = cuda.blockIdx.x * TILE - 1
    y0 = cuda.blockIdx.y * TILE - 1
    for j in range(ty, SH, TILE):
        yy = min(max(y0 + j, 0), H - 1)
        for i in range(tx, SH, TILE):
            xx = min(max(x0 + i, 0), W - 1)
            tile[j, i] = src[yy, xx]
    cuda.syncthreads()
    x = cuda.blockIdx.x * TILE + tx
    y = cuda.blockIdx.y * TILE + ty
    if x < W and y < H:
        a, b, c = tile[ty, tx], tile[ty, tx + 1], tile[ty, tx + 2]
        d, f = tile[ty + 1, tx], tile[ty + 1, tx + 2]
        g, h, i2 = tile[ty + 2, tx], tile[ty + 2, tx + 1], tile[ty + 2, tx + 2]
        gx = (c + 2 * f + i2) - (a + 2 * d + g)
        gy = (g + 2 * h + i2) - (a + 2 * b + c)
        dst[y, x] = min(math.sqrt(gx * gx + gy * gy), 255.0)
 
@cuda.jit
def k_median3(src, dst, H, W):
    tile = cuda.shared.array((SH, SH), float32)
    tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
    x0 = cuda.blockIdx.x * TILE - 1
    y0 = cuda.blockIdx.y * TILE - 1
    for j in range(ty, SH, TILE):
        yy = min(max(y0 + j, 0), H - 1)
        for i in range(tx, SH, TILE):
            xx = min(max(x0 + i, 0), W - 1)
            tile[j, i] = src[yy, xx]
    cuda.syncthreads()
    x = cuda.blockIdx.x * TILE + tx
    y = cuda.blockIdx.y * TILE + ty
    if x < W and y < H:
        win = cuda.local.array(9, float32)
        k = 0
        for dy in range(3):
            for dx in range(3):
                win[k] = tile[ty + dy, tx + dx]
                k += 1
        for i in range(1, 9):
            v = win[i]
            j = i - 1
            while j >= 0 and win[j] > v:
                win[j + 1] = win[j]
                j -= 1
            win[j + 1] = v
        dst[y, x] = win[4]
 
@cuda.jit
def k_hist(src, hist, H, W):
    sh = cuda.shared.array(256, int32)  # per-block histogram cuts global atomic contention
    t = cuda.threadIdx.y * TILE + cuda.threadIdx.x
    if t < 256:
        sh[t] = 0
    cuda.syncthreads()
    x, y = cuda.grid(2)
    if x < W and y < H:
        cuda.atomic.add(sh, min(max(int(src[y, x]), 0), 255), 1)
    cuda.syncthreads()
    if t < 256:
        cuda.atomic.add(hist, t, sh[t])
 
@cuda.jit
def k_apply_lut(src, lut, dst, H, W):
    x, y = cuda.grid(2)
    if x < W and y < H:
        dst[y, x] = lut[min(max(int(src[y, x]), 0), 255)]
 
_conv_cache = {}
def get_conv(R):
    if R not in _conv_cache:
        _conv_cache[R] = make_conv_kernel(R)
    return _conv_cache[R]

In [ ]:
def run_gpu(rgb, R=2, keep_outputs=True):
    H, W, _ = rgb.shape
    grid = (math.ceil(W / TILE), math.ceil(H / TILE))
    conv = get_conv(R)
    w = gaussian_weights(R)
 
    ev = [cuda.event(timing=True) for _ in range(4)]
    ev[0].record()
    d_rgb = cuda.to_device(rgb)                       # H2D over PCIe
    d_w = cuda.to_device(w)
    ev[1].record()
 
    d_gray = cuda.device_array((H, W), np.float32)
    d_blur = cuda.device_array((H, W), np.float32)
    d_edge = cuda.device_array((H, W), np.float32)
    d_med = cuda.device_array((H, W), np.float32)
    d_heq = cuda.device_array((H, W), np.float32)
    d_hist = cuda.to_device(np.zeros(256, np.int32))
 
    k_gray[grid, TPB](d_rgb, d_gray, H, W)
    conv[grid, TPB](d_gray, d_blur, d_w, H, W)
    k_sobel[grid, TPB](d_blur, d_edge, H, W)
    k_median3[grid, TPB](d_gray, d_med, H, W)
    k_hist[grid, TPB](d_gray, d_hist, H, W)
    hist = d_hist.copy_to_host()                      # 1 KB, then LUT on host
    cdf = np.cumsum(hist)
    cmin = cdf[np.nonzero(cdf)[0][0]]
    lut = (np.maximum(cdf - cmin, 0) / max(1, H * W - cmin) * 255).astype(np.float32)
    k_apply_lut[grid, TPB](d_gray, cuda.to_device(lut), d_heq, H, W)
    ev[2].record()
 
    outs = None
    if keep_outputs:
        outs = dict(gray=d_gray.copy_to_host(), blur=d_blur.copy_to_host(),
                    edges=d_edge.copy_to_host(), median=d_med.copy_to_host(),
                    heq=d_heq.copy_to_host())          # D2H
    ev[3].record()
    ev[3].synchronize()
    t = dict(h2d=cuda.event_elapsed_time(ev[0], ev[1]),
             kernels=cuda.event_elapsed_time(ev[1], ev[2]),
             d2h=cuda.event_elapsed_time(ev[2], ev[3]))
    t["total"] = t["h2d"] + t["kernels"] + t["d2h"]
    return outs, t

In [ ]:
def show_results(path, R=2):
    rgb = load_rgb(path)
    run_gpu(rgb, R); run_cpu(CPU_SEQ, rgb, R)          # warm-up (JIT compile)
    gpu_out, t = run_gpu(rgb, R)
    cpu_out = run_cpu(CPU_SEQ, rgb, R)
 
    panels = [("Original", rgb), ("Grayscale", gpu_out["gray"]),
              (f"Gaussian (k={2*R+1})", gpu_out["blur"]), ("Sobel edges", gpu_out["edges"]),
              ("Median 3x3", gpu_out["median"]), ("Hist. equalized", gpu_out["heq"])]
    fig, axs = plt.subplots(2, 3, figsize=(15, 7))
    for ax, (title, img) in zip(axs.ravel(), panels):
        if img.ndim == 3: ax.imshow(img)
        else: ax.imshow(img, cmap="gray", vmin=0, vmax=255)
        ax.set_title(title); ax.axis("off")
    plt.tight_layout(); plt.show()
 
    for k in ("blur", "edges", "median", "heq"):       # correctness check vs CPU
        print(f"{k:7s} max |GPU - CPU| = {np.abs(gpu_out[k] - cpu_out[k]).max():.3f}")
    print({k: f"{v:.2f} ms" for k, v in t.items()})
 
show_results(paths[0])

In [ ]:
def timeit(fn, reps=3, warm=True):
    if warm:
        fn()                                           # warm-up
    best = 1e9
    for _ in range(reps):
        s = time.perf_counter(); fn(); best = min(best, time.perf_counter() - s)
    return best * 1e3
 
RESOLUTIONS = [(512, 768), (1024, 1536), (2048, 3072), (4096, 6144)]
RADII = [1, 2, 4, 7]                                    # kernel sizes 3, 5, 9, 15
tiny = load_rgb(paths[0], (64, 96))                     # compile all CPU/GPU code up front
for R_ in RADII:
    run_cpu(CPU_SEQ, tiny, R_); run_cpu(CPU_OMP, tiny, R_); run_gpu(tiny, R_)
rows = []
for res in RESOLUTIONS:
    rgb = load_rgb(paths[0], res)
    for R in RADII:
        seq = timeit(lambda: run_cpu(CPU_SEQ, rgb, R), reps=1 if res[0] >= 2048 else 3, warm=res[0] < 2048)
        omp = timeit(lambda: run_cpu(CPU_OMP, rgb, R), reps=2 if res[0] >= 2048 else 3, warm=res[0] < 2048)
        t = min((run_gpu(rgb, R, keep_outputs=True)[1] for _ in range(3)), key=lambda d: d["total"])
        rows.append(dict(res=f"{res[1]}x{res[0]}", mpix=res[0] * res[1] / 1e6, k=2 * R + 1,
                         seq=seq, omp=omp, gpu_kernel=t["kernels"], gpu_total=t["total"],
                         h2d=t["h2d"], d2h=t["d2h"]))
        r = rows[-1]
        print(f"{r['res']:>10} k={r['k']:>2} seq={seq:8.1f} omp={omp:8.1f} "
              f"gpu(kernels)={r['gpu_kernel']:6.2f} gpu(total)={r['gpu_total']:6.2f} ms")
 
try:
    import pandas as pd
    df = pd.DataFrame(rows)
    df["speedup_vs_seq"] = df.seq / df.gpu_total
    df["speedup_vs_omp"] = df.omp / df.gpu_total
    df["throughput_MPix_s"] = df.mpix / (df.gpu_total / 1e3)
    display(df.round(2)) if "display" in globals() else print(df.round(2))
except ImportError:
    df = None

In [ ]:
if df is not None:
    fig, axs = plt.subplots(1, 3, figsize=(17, 4.5))
    for k, g in df.groupby("k"):
        axs[0].plot(g.mpix, g.speedup_vs_seq, "o-", label=f"k={k}")
        axs[1].plot(g.mpix, g.speedup_vs_omp, "o-", label=f"k={k}")
    axs[0].set(title="GPU speedup vs sequential CPU", xlabel="Megapixels", ylabel="x (incl. transfers)")
    axs[1].set(title="GPU speedup vs OpenMP-style CPU", xlabel="Megapixels", ylabel="x (incl. transfers)")
    g = df[df.k == 5]
    axs[2].bar(g.res, g.h2d, label="H2D"); axs[2].bar(g.res, g.gpu_kernel, bottom=g.h2d, label="Kernels")
    axs[2].bar(g.res, g.d2h, bottom=g.h2d + g.gpu_kernel, label="D2H")
    axs[2].set(title="GPU time breakdown (k=5)", ylabel="ms")
    for a in axs: a.legend(); a.grid(alpha=.3)
    plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
try:
    gpu_label = f"GPU ({cuda.get_current_device().name.decode()}, incl. transfers)"
except Exception:
    gpu_label = "GPU (incl. transfers)"
 
def fmt(ms):
    if ms >= 1000: return f"{ms/1000:.1f} s"
    if ms >= 100:  return f"{ms:.0f} ms"
    return f"{ms:.1f} ms"
 
APPROACHES = [("seq", "Sequential CPU", "#5b6b7f"),
              ("omp", f"Multi-threaded CPU ({nb.config.NUMBA_NUM_THREADS} threads)", "#e0a030"),
              ("gpu_total", gpu_label, "#2a9d6f")]
 
print(df[["res", "k", "seq", "omp", "gpu_total"]].round(1).rename(
    columns={"seq": "seq_ms", "omp": "omp_ms", "gpu_total": "gpu_ms"}).to_string(index=False))
 
# Figure 1: one panel per kernel size, grouped bars per image size, log scale
res_order = list(dict.fromkeys(df.res))
ymax = df[[a[0] for a in APPROACHES]].max().max()
ymin = df[[a[0] for a in APPROACHES]].min().min()
fig, axs = plt.subplots(2, 2, figsize=(15, 9), sharey=True)
for ax, (k, g) in zip(axs.ravel(), df.groupby("k")):
    g = g.set_index("res").loc[res_order]
    x, w = np.arange(len(g)), 0.27
    for j, (col, label, c) in enumerate(APPROACHES):
        bars = ax.bar(x + (j - 1) * w, g[col], w, label=label, color=c)
        for b, v in zip(bars, g[col]):
            ax.text(b.get_x() + b.get_width() / 2, v * 1.1, fmt(v), ha="center",
                    va="bottom", fontsize=7.5, rotation=90)
    ax.set_yscale("log"); ax.set_ylim(ymin * 0.5, ymax * 4)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{r}\n{m:.1f} MP" for r, m in zip(g.index, g.mpix)])
    ax.set_title(f"Gaussian kernel k={k} ({k}x{k})")
    ax.set_ylabel("Total time (ms, log scale)")
    ax.grid(axis="y", alpha=.3, which="both")
h, l = axs[0, 0].get_legend_handles_labels()
fig.legend(h, l, loc="upper center", bbox_to_anchor=(0.5, 0.955), ncol=3, frameon=False, fontsize=11)
fig.suptitle("Time taken by each approach (lower is better)", y=1.0, fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig("time_comparison_by_size.png", dpi=200, bbox_inches="tight"); plt.show()
 
# Figure 2: largest image, linear scale, so the real size of the gap is visible
big = df[df.mpix == df.mpix.max()].sort_values("k")
fig, ax = plt.subplots(figsize=(10, 5.5))
x, w = np.arange(len(big)), 0.27
for j, (col, label, c) in enumerate(APPROACHES):
    bars = ax.bar(x + (j - 1) * w, big[col] / 1000, w, label=label, color=c)
    for b, v in zip(bars, big[col]):
        ax.text(b.get_x() + b.get_width() / 2, v / 1000, fmt(v), ha="center", va="bottom", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels([f"k={k} ({k}x{k})" for k in big.k])
ax.set_ylabel("Total time (seconds, linear scale)")
ax.set_title(f"Time to process one {big.res.iloc[0]} image (lower is better)")
ax.legend(frameon=False); ax.grid(axis="y", alpha=.3)
plt.tight_layout()
plt.savefig("time_comparison_largest.png", dpi=200, bbox_inches="tight"); plt.show()